In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('./data/ObesityDataSet_raw_and_data_sinthetic.csv')
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [5]:
#On veut prédire la colonne NObeyesdad
df.NObeyesdad.value_counts()

# c'est une un problème de classification multi-classes avec 7 classes différentes avec une ordinalité potentielle (Insufficient → Normal → Overweight I/II → Obesity I/II/III)

NObeyesdad
Obesity_Type_I         351
Obesity_Type_III       324
Obesity_Type_II        297
Overweight_Level_I     290
Overweight_Level_II    290
Normal_Weight          287
Insufficient_Weight    272
Name: count, dtype: int64

In [ ]:
len(df)

2111

In [41]:
print(df.MTRANS.value_counts()) # 5 catégories
print(df.Gender.value_counts()) # passer en 0/1
print(df.Age.value_counts())
print(df.Height.value_counts())
print(df.Weight.value_counts())
print(df.family_history_with_overweight.value_counts()) # passer en 0/1
print(df.FAVC.value_counts()) #  passer en 0/1
print(df.FCVC.value_counts())
print(df.NCP.value_counts())
print(df.CAEC.value_counts()) # 4 catégories
print(df.SMOKE.value_counts()) # passer en 0/1 # sous représentation des fumeurs
print(df.SCC.value_counts()) # passer en 0/1
print(df.FAF.value_counts())
print(df.TUE.value_counts())
print(df.CH2O.value_counts())
print(df.CALC.value_counts()) # 4 catégories


MTRANS
Public_Transportation    1580
Automobile                457
Walking                    56
Motorbike                  11
Bike                        7
Name: count, dtype: int64
Gender
Male      1068
Female    1043
Name: count, dtype: int64
Age
18.000000    128
26.000000    101
21.000000     96
23.000000     89
19.000000     59
            ... 
21.680123      1
24.469756      1
25.127910      1
25.986368      1
23.761970      1
Name: count, Length: 1402, dtype: int64
Height
1.700000    60
1.650000    50
1.600000    43
1.750000    39
1.620000    36
            ..
1.643421     1
1.640535     1
1.626483     1
1.645990     1
1.631547     1
Name: count, Length: 1574, dtype: int64
Weight
80.000000     59
70.000000     43
50.000000     42
75.000000     40
60.000000     37
              ..
111.939983     1
111.555967     1
111.357062     1
111.922491     1
102.174953     1
Name: count, Length: 1525, dtype: int64
family_history_with_overweight
yes    1726
no      385
Name: count, dtype: in

In [43]:
print(df.isna().sum())

Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64


In [54]:
print(df.dtypes)


Gender                             object
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight     object
FAVC                               object
FCVC                              float64
NCP                               float64
CAEC                               object
SMOKE                              object
CH2O                              float64
SCC                                object
FAF                               float64
TUE                               float64
CALC                               object
MTRANS                             object
NObeyesdad                         object
dtype: object


In [11]:
%pip install scikit-learn
%pip install lightgbm


  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)

   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]
   ---------------------------------------- 0/2 [joblib]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix, make_scorer, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate
import lightgbm as lgb
import warnings
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier

In [ ]:
# On ajoute le BMI qui est une relation non linéaire pour améliorer le modèle
def add_basic_features(df):
    if {'Weight', 'Height'}.issubset(df.columns):
        df['BMI'] = df['Weight'] / (df['Height']**2)
    return df
df = add_basic_features(df)

X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist() #respecte bien ce qui a été observé avec les values_counts()

preproc = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=10), cat_cols)
    ],
    remainder='drop'
)

In [ ]:
warnings.filterwarnings('ignore')
models = {
    'LogisticRegression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, penalty='l2', C=1.0, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=100, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0, verbosity=-1, random_state=42),
    # 'MLP': MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', batch_size=32, max_iter=2000, early_stopping=True, learning_rate_init=0.001, random_state=42)
}
results = {}
scoring = {                     # métriques à justifier dans le rapport : on veut accuracy mais pas que ! on veut vérifier aussi l'erreur en distance entre les classes prédites et réelles via l'ordinalité des classes à prédire (cf cellule suivante)
    'accuracy': 'accuracy',
    'macro_f1': make_scorer(f1_score, average='macro'),
    'balanced_accuracy': 'balanced_accuracy',
    'mcc': make_scorer(matthews_corrcoef)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in models.items():
    pipe_model = Pipeline([
        ('preproc', preproc),
        ('clf', clf)
    ])
    print(f'\n--- {name} ---')
    cv_results = cross_validate(pipe_model, X, y, cv=cv, scoring=scoring, return_train_score=False, error_score='raise')
    for score in scoring:
        print(f"{score}: {cv_results['test_' + score].mean():.3f} ± {cv_results['test_' + score].std():.3f}")
    # Fit and show confusion matrix on hold-out set
    pipe_model.fit(X_train, y_train)
    y_pred = pipe_model.predict(X_test)
    print('Confusion matrix (test set):')
    print(confusion_matrix(y_test, y_pred))


--- LogisticRegression ---
accuracy: 0.879 ± 0.012
macro_f1: 0.875 ± 0.012
balanced_accuracy: 0.877 ± 0.012
mcc: 0.859 ± 0.014
Confusion matrix (test set):
[[54  0  0  0  0  0  0]
 [ 5 43  0  0  0 10  0]
 [ 0  0 64  3  0  0  3]
 [ 0  0  2 58  0  0  0]
 [ 0  0  0  1 64  0  0]
 [ 0  8  0  0  0 43  7]
 [ 0  2  7  0  0  5 44]]

--- RandomForest ---
accuracy: 0.879 ± 0.012
macro_f1: 0.875 ± 0.012
balanced_accuracy: 0.877 ± 0.012
mcc: 0.859 ± 0.014
Confusion matrix (test set):
[[54  0  0  0  0  0  0]
 [ 5 43  0  0  0 10  0]
 [ 0  0 64  3  0  0  3]
 [ 0  0  2 58  0  0  0]
 [ 0  0  0  1 64  0  0]
 [ 0  8  0  0  0 43  7]
 [ 0  2  7  0  0  5 44]]

--- RandomForest ---
accuracy: 0.917 ± 0.013
macro_f1: 0.916 ± 0.013
balanced_accuracy: 0.916 ± 0.013
mcc: 0.904 ± 0.015
accuracy: 0.917 ± 0.013
macro_f1: 0.916 ± 0.013
balanced_accuracy: 0.916 ± 0.013
mcc: 0.904 ± 0.015
Confusion matrix (test set):
[[51  3  0  0  0  0  0]
 [ 0 49  0  0  0  8  1]
 [ 0  2 65  0  0  0  3]
 [ 0  0  1 59  0  0  0]
 [ 0  1

In [ ]:
# Calcul de la MAE ordinale sur le test set pour chaque modèle
ordinal_mapping = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}
print("\n--- MAE ordinale sur le test set ---")
for name, clf in models.items():
    pipe_model = Pipeline([
        ('preproc', preproc),
        ('clf', clf)
    ])
    pipe_model.fit(X_train, y_train)
    y_pred = pipe_model.predict(X_test)
    y_true_ord = y_test.map(ordinal_mapping)
    y_pred_ord = pd.Series(y_pred).map(ordinal_mapping)
    mae_ord = (y_true_ord - y_pred_ord).abs().mean()
    print(f"{name}: MAE ordinale = {mae_ord:.3f}")


# conclusion : lorsqu'une erreur est commise elle est en moyenne de l'ordre de 2 classes


--- MAE ordinale sur le test set ---
LogisticRegression: MAE ordinale = 2.032
LogisticRegression: MAE ordinale = 2.032
RandomForest: MAE ordinale = 2.011
RandomForest: MAE ordinale = 2.011
LightGBM: MAE ordinale = 2.032
LightGBM: MAE ordinale = 2.032


1. Régression Logistique Multinomiale
Critère utilisé :

Accuracy (précision globale)
Macro F1-score
Balanced accuracy
MCC (Matthews Correlation Coefficient)
Matrice de confusion
Pertinence :

L’accuracy mesure la proportion de bonnes prédictions, mais peut être trompeuse si les classes sont déséquilibrées.
Le macro F1-score fait la moyenne des F1-scores de chaque classe, ce qui donne le même poids à toutes les classes, même les minoritaires : il est donc pertinent pour un problème multi-classes avec des déséquilibres.
La balanced accuracy corrige l’accuracy en tenant compte du déséquilibre des classes.
Le MCC est robuste aux déséquilibres et synthétise la qualité de la classification sur toutes les classes.
La matrice de confusion permet d’identifier les classes confondues.
2. Random Forest
Critère utilisé :

Identique à la régression logistique : accuracy, macro F1-score, balanced accuracy, MCC, matrice de confusion.
Pertinence :

Les mêmes arguments s’appliquent.
Ces métriques permettent de comparer la capacité du Random Forest à bien classer toutes les classes, y compris les minoritaires, et à éviter le surapprentissage.
3. LightGBM
Critère utilisé :

Identique : accuracy, macro F1-score, balanced accuracy, MCC, matrice de confusion.
Pertinence :

LightGBM étant un modèle puissant, il est important de surveiller non seulement l’accuracy mais aussi les scores macro et MCC pour s’assurer qu’il ne sur-apprend pas sur les classes majoritaires.
La matrice de confusion permet de vérifier que les classes rares ne sont pas négligées.
4. MLPClassifier (non évalué ici à cause d’un bug)
Critère recommandé :

Les mêmes métriques : accuracy, macro F1-score, balanced accuracy, MCC, matrice de confusion.
Pertinence :

Pour un réseau de neurones, il est crucial d’utiliser des métriques robustes aux déséquilibres, car il peut facilement sur-apprendre ou ignorer des classes minoritaires.

In [ ]:
# Si on fait que un seul modèle à la fois on peut faire ça comme ça : (exemple de lightgbm)

# Modèle LightGBM avec régularisation renforcée pour limiter l'overfitting
gbm = LGBMClassifier(
    objective='multiclass',
    num_class=7,
    n_estimators=100,  # Réduit pour limiter l'overfitting
    learning_rate=0.05,
    max_depth=4,       # Limité (-1 = illimité)
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,     # Augmenté (avant 0.1) pour limiter l'overfitting
    reg_lambda=1.0,     # Augmenté (avant 0.3) pour limiter l'overfitting
    verbosity=-1 #on évite les warnings de lightgbm
)

pipe = Pipeline([
    ('preproc', preproc),
    ('gbm', gbm)
])


# Entraînement + calibration (isotonic recommandée si jeu suffisant)
pipe.fit(X_train, y_train)
calibrated = CalibratedClassifierCV(pipe, method='isotonic', cv=3)
calibrated.fit(X_train, y_train)

y_pred = calibrated.predict(X_test)
y_proba = calibrated.predict_proba(X_test)

print(classification_report(y_test, y_pred, digits=3))
print(confusion_matrix(y_test, y_pred))

